In [1]:
import kagglehub
import os
import pandas as pd



# Download latest version
path = kagglehub.dataset_download("jeromeblanchet/recipeqa-nlp-dataset")

train = pd.read_json(os.path.join(path, "train recipeqa.json"))
test = pd.read_json(os.path.join(path, "test recipeqa.json"))
val = pd.read_json(os.path.join(path, "val recipeqa.json"))
train.head()
print(train.iloc[0]["data"])
print(train.iloc[0]["data"].keys())


100%|██████████| 2.55G/2.55G [00:36<00:00, 75.9MB/s]

Extracting files...


{'recipe_id': 'how-to-make-halal-vanilla-extract', 'context_modality': ['body', 'title', 'videos'], 'split': 'train', 'context': [{'body': '3 until 5 whole vanilla beans250 gram of vegetable glycerin food gradeEvery 100 gram of vanilla beans have 35 until 40 of whole vanilla beans', 'id': 1, 'videos': [], 'title': 'Ingredients Halal Vanilla Extract'}, {'body': 'Scrape Vanilla Beans and get the seeds into vegetable glycerin', 'id': 2, 'videos': [], 'title': 'Scrape Vanilla Beans'}, {'body': 'Vanilla Beans Seed and Vegetable Glycerin', 'id': 3, 'videos': [], 'title': 'Vegetable Glycerin and Vanilla Beans'}, {'body': 'Whole Vanilla Beans put in a bottle with seeds and vegetable glycerin', 'id': 4, 'videos': [], 'title': 'Vanilla Beans Can Use With Vegetable Glycerin'}], 'choice_list': ['how-to-make-halal-vanilla-extract_1_0.jpg', 'how-to-make-halal-vanilla-extract_2_0.jpg', 'how-to-make-halal-vanilla-extract_3_0.jpg', '-1-23-and-it-was-delicious-i-ate-some-pie-a_4_3.jpg'], 'answer': 3, 'q

In [4]:
from tqdm.auto import tqdm
def extract_textual_samples(df_sample, split_name):
  limit = 0
  samples=[]
  for _,row in tqdm( df_sample.iterrows(), total=len(df_sample), desc= f"Processing {split_name}"):
      row = row["data"]
      if row["task"] in ["textual_cloze", "temporal_ordering", "step_identification", "ingredient_matching"]:
        full_context = " ".join(step["body"] for step in row["context"])
        question = row.get("question_text")  # safely get question_text
        if not isinstance(question, str) or not question.strip():  # if empty or missing
            question = row["question"][0] if isinstance(row["question"], list) else row["question"]
        samples.append({
          "split": split_name,
          "title": row["recipe_id"],
          "context": full_context,
          "question": question if question else row["question_text"],
          "choices": row["choice_list"],
          "answer_idx": row["answer"],
          "correct_answer": row["choice_list"][row["answer"]] if row["answer"] < len(row["choice_list"]) else "N/A",
          "task": row["task"],
          "context_length_words": len(full_context.split())})
  return pd.DataFrame(samples)

train_text = extract_textual_samples(train, "train")
test_text = extract_textual_samples(test, "test")
val_text = extract_textual_samples(val, "val")

df = pd.concat([train_text, test_text, val_text], ignore_index=True)
print(f"\nتعداد کل سوال‌های متنی (textual): {len(df)}")
print(df['split'].value_counts())
print(df['task'].value_counts())

# آمار طول context (این عدد طلاییه!)
print("\nآمار طول دستور پخت‌ها (کلمات):")
print(df["context_length_words"].describe())

print(f"\nدرصد دستور پخت‌های بالای ۴۰۰ کلمه: {100*(df['context_length_words'] > 400).mean():.2f}%")
print(f"بلندترین دستور پخت: {df['context_length_words'].max()} کلمه")

Processing train:   0%|          | 0/29657 [00:00<?, ?it/s]

Processing test:   0%|          | 0/3567 [00:00<?, ?it/s]

Processing val:   0%|          | 0/3562 [00:00<?, ?it/s]


تعداد کل سوال‌های متنی (textual): 9761
split
train    7837
test      963
val       961
Name: count, dtype: int64
task
textual_cloze    9761
Name: count, dtype: int64

آمار طول دستور پخت‌ها (کلمات):
count     9761.000000
mean       489.555476
std        486.186792
min         13.000000
25%        222.000000
50%        365.000000
75%        600.000000
max      16605.000000
Name: context_length_words, dtype: float64

درصد دستور پخت‌های بالای ۴۰۰ کلمه: 45.30%
بلندترین دستور پخت: 16605 کلمه


In [5]:
# نصب فقط یک بار
!pip install -q rank_bm25 scikit-learn tqdm sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.7 MB/s eta 0:00:00


In [16]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments, T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch
from sentence_transformers import SentenceTransformer
import faiss
from torch.utils.data import Dataset

class RecipeQADataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        context = row["context"]
        question = row["question"]
        choices = row["choices"]
        label = row["answer_idx"]

        # Encode for multiple choice
        encoding = self.tokenizer(
            [f"{question} {context}" for _ in choices],  # repeated context
            choices,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        # Remove batch dimension so Trainer can collate
        item = {k: encoding[k].squeeze(0) for k in encoding}
        item["labels"] = torch.tensor(label)
        return item

class RecipeQAT5Dataset(Dataset):
    def __init__(self, df, tokenizer, max_input_length=512, max_target_length=64):
        self.df = df
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        input_text = f"question: {row['question']} context: {row['context']}"
        target_text = row["correct_answer"]

        # Encode input
        input_enc = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Encode target
        target_enc = self.tokenizer(
            target_text,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = target_enc["input_ids"].squeeze()
        labels[labels == t5_tokenizer.pad_token_id] = -100

        item = {
            "input_ids": input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels": labels
        }

        return item


device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("/content/drive/MyDrive/result/checkpoint-5880")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

model.to(device)
t5_model.to(device)

df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val = df[df["split"] == "val"].reset_index(drop=True)

train_dataset = RecipeQADataset(df_train, tokenizer)
val_dataset   = RecipeQADataset(df_val, tokenizer)
train_t5_dataset = RecipeQAT5Dataset(df_train, t5_tokenizer)
val_t5_dataset   = RecipeQAT5Dataset(df_val, t5_tokenizer)

arg = TrainingArguments(
    output_dir = "./result",
    num_train_epochs = 6,
    per_device_train_batch_size = 6,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_dir = "./logs",
    load_best_model_at_end=True,
    save_total_limit=2,
    learning_rate = 3e-5,
    weight_decay = 0.01,
)

trainer = Trainer(
    model = model,
    args = arg,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

training_args_t5 = Seq2SeqTrainingArguments(
    output_dir="./resultT5",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    predict_with_generate=True,
    logging_dir="./logsT5"
)

trainerT5 = Seq2SeqTrainer(
    model=t5_model,
    args=training_args_t5,
    train_dataset=train_t5_dataset,
    eval_dataset=val_t5_dataset,
)

#trainerT5.train()
#trainer.train()

retriever = SentenceTransformer("all-MiniLM-L6-v2")
t5_model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/resultT5/checkpoint-5880").to(device)
contexts = df["context"].tolist()
embeddings = retriever.encode(contexts, normalize_embeddings=True, batch_size=32)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def encode_mc(context, question, choices):
  text_pair = [f"{question} {context}" for _ in choices]
  encoding = tokenizer(text_pair, choices, padding=True, truncation=True, return_tensors="pt")

  for k in encoding:
    encoding[k] = encoding[k].unsqueeze(0).to(device)
  return encoding

#Here is the test to check sync between colab and github
def predict_mc(context, question, choices, threshold = 0.5):
  inputs = encode_mc(context, question, choices)
  outputs = model(**inputs)
  logits = outputs.logits
  prob = torch.softmax(logits, dim=1)
  pred_idx = torch.argmax(logits).item()

  conf = prob.max().item()
  return pred_idx, conf

def generate_answer(context, question, max_len=50):
  input_text = f"question: {question} context: {context}"
  input_ids = t5_tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
  output_ids = t5_model.generate(input_ids, max_length=max_len)
  answer = t5_tokenizer.decode(output_ids[0], skip_special_tokens = True)
  return answer

def answer_question(question, choices , k=3, threshold = 0.5):
  q_emb = retriever.encode([question], normalize_embeddings=True)
  scores, indices = index.search(q_emb, k)
  best_context = df.iloc[indices[0]]["context"]
  pred_idx, conf = predict_mc(best_context, question, choices)

  if conf >= threshold:
    return choices[pred_idx], "Bert"
  else:
    answer = generate_answer(best_context, question)
    return answer, "T5"

correct_total = 0

for idx, row in tqdm(df_val.iterrows(), total=len(df_val)):
    answer, source = answer_question(row["question"], row["choices"])

    if source == "Bert":
        if answer == row["correct_answer"]:
            correct_total += 1
    else:  # T5 fallback
        if answer.strip().lower() == row["correct_answer"].strip().lower():
            correct_total += 1

overall_accuracy = correct_total / len(df_val)
print(f"Overall System Accuracy: {overall_accuracy:.4f}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

100%|██████████| 961/961 [00:43<00:00, 22.28it/s]

Overall System Accuracy: 0.3538


In [17]:
import re
from collections import Counter

def normalize_text(s):
    s = s.lower()
    s = re.sub(r'[^\w\s]', '', s)
    return s

def compute_f1(pred, truth):
    pred_tokens = normalize_text(pred).split()
    truth_tokens = normalize_text(truth).split()

    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return f1

def semantic_similarity(pred, truth):
    emb = retriever.encode([pred, truth], normalize_embeddings=True)
    return float((emb[0] @ emb[1]))

from tqdm import tqdm

exact_match = 0
f1_total_bert = 0
f1_total_t5 = 0
sim_total_bert = 0
sim_total_t5 = 0
n_bert_response = 0
n_t5_response = 0

for idx, row in tqdm(df_val.iterrows(), total=len(df_val)):
    answer, source = answer_question(row["question"], row["choices"])
    truth = row["correct_answer"]

    if answer.strip().lower() == truth.strip().lower():
        exact_match += 1
    if source == "Bert":
      f1_total_bert += compute_f1(answer, truth)
      sim_total_bert += semantic_similarity(answer, truth)
      n_bert_response += 1
    else:  # T5 fallback
        f1_total_t5 += compute_f1(answer, truth)
        sim_total_t5 += semantic_similarity(answer, truth)
        n_t5_response += 1

n = len(df_val)

print("Exact Match:", exact_match / n)
print("Average F1 for BERT:", f1_total_bert / n_bert_response)
print("Average Semantic Similarity for BERT:", sim_total_bert / n_bert_response)
print("Average F1 for T5:", f1_total_t5 / n_t5_response)
print("Average Semantic Similarity for T5:", sim_total_t5 / n_t5_response)

100%|██████████| 961/961 [00:51<00:00, 18.68it/s]

Exact Match: 0.3537981269510926
Average F1 for BERT: 0.43982560333392595
Average Semantic Similarity for BERT: 0.5437292091193086
Average F1 for T5: 0.059920634920634916
Average Semantic Similarity for T5: 0.2517504450140728


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
